# PointChess 4-Billion Position NNUE Training (Kaggle 2x NVIDIA T4 GPUs)

This notebook trains PointChess's **Modern HalfKAv2** NNUE neural network with **SCReL** and **AMP fp16** across **2x NVIDIA Tesla T4 GPUs** using a 4-billion position streaming architecture.

### Accelerator Setup:
Go to **Settings (right sidebar) -> Accelerator -> GPU T4 x2**.

In [ ]:
# Step 1: Verify 2x Tesla T4 GPUs
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f" GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# Step 2: Unpack or Clone PointChess Repository
import os
if not os.path.exists("src"):
    !git clone https://github.com/hemu-create/PointChess.git pointchess_repo
    !cp -r pointchess_repo/* .
!mkdir -p build && cd build && cmake -DCMAKE_BUILD_TYPE=Release .. && make -j$(nproc)

In [ ]:
# Step 3: Run High-Throughput 4-Billion Position Trainer with PyTorch DDP
# Streams positions across 2x T4 GPUs at >50,000 pos/sec
!cd training && torchrun --nproc_per_node=2 train_4b.py \
    --data_file /kaggle/input/pointchess-200k-dataset/pointchess_200k_tactical.txt \
    --max_positions 4000000000 \
    --batch_size 4096 \
    --lr 2e-3 \
    --output_dir /kaggle/working/checkpoints_4b

In [ ]:
# Step 4: Export Master .pchess and .pnet Network Files
!cd training && python3 export.py \
    --checkpoint /kaggle/working/checkpoints_4b/best_model.pt \
    --output_pchess /kaggle/working/pointchess.pchess \
    --output_pnet /kaggle/working/nnue.pnet

!ls -lh /kaggle/working/pointchess.pchess /kaggle/working/nnue.pnet